In [1]:
# Step 18: Final Model and Kaggle Submission

import numpy as np
import pandas as pd

from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import Ridge


# 1. Find competition files
train_files = list(
    Path("/kaggle/input").rglob("train.csv")
)

test_files = list(
    Path("/kaggle/input").rglob("test.csv")
)

if not train_files or not test_files:
    raise FileNotFoundError(
        "Dataset not found. Check Add Input."
    )

train_raw = pd.read_csv(train_files[0])
test_raw = pd.read_csv(test_files[0])

print("Training data:", train_raw.shape)
print("Test data:", test_raw.shape)


# 2. Feature engineering function
def add_features(df):
    df = df.copy()

    df["TotalSF"] = (
        df["TotalBsmtSF"].fillna(0)
        + df["1stFlrSF"].fillna(0)
        + df["2ndFlrSF"].fillna(0)
    )

    df["TotalBathrooms"] = (
        df["FullBath"].fillna(0)
        + 0.5 * df["HalfBath"].fillna(0)
        + df["BsmtFullBath"].fillna(0)
        + 0.5 * df["BsmtHalfBath"].fillna(0)
    )

    porch_cols = [
        "OpenPorchSF",
        "3SsnPorch",
        "EnclosedPorch",
        "ScreenPorch",
        "WoodDeckSF"
    ]

    df["TotalPorchSF"] = (
        df[porch_cols].fillna(0).sum(axis=1)
    )

    df["HouseAge"] = (
        df["YrSold"] - df["YearBuilt"]
    )

    df["RemodAge"] = (
        df["YrSold"] - df["YearRemodAdd"]
    )

    df["OverallQual_SF"] = (
        df["OverallQual"].fillna(0)
        * df["GrLivArea"].fillna(0)
    )

    return df


# 3. Apply the same features to train and test
train = add_features(train_raw)
test = add_features(test_raw)


# 4. Separate target and features
X = train.drop(
    columns=["SalePrice", "Id"],
    errors="ignore"
)

y = np.log1p(train["SalePrice"])

X_test = test.drop(
    columns=["Id"],
    errors="ignore"
)


# 5. Identify numerical and categorical columns
numeric_features = X.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object", "category"]
).columns.tolist()


# 6. Create preprocessing pipelines
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="constant",
            fill_value="Missing"
        )
    ),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])


# 7. Build and train the selected model
final_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", Ridge(alpha=10))
])

final_model.fit(X, y)

print("Final model trained successfully!")


# 8. Predict test prices
predictions_log = final_model.predict(X_test)

predictions = np.expm1(predictions_log)
predictions = np.maximum(predictions, 0)


# 9. Create Kaggle submission
submission = pd.DataFrame({
    "Id": test_raw["Id"],
    "SalePrice": predictions
})

output_path = "/kaggle/working/submission.csv"

submission.to_csv(
    output_path,
    index=False
)


# 10. Verify the submission
print("\nSubmission file saved:", output_path)
print("Submission shape:", submission.shape)
print("Columns:", submission.columns.tolist())

display(submission.head())

assert submission["SalePrice"].notna().all()
assert (submission["SalePrice"] >= 0).all()
assert submission["Id"].equals(test_raw["Id"])

print("\nSubmission checks passed!")

Training data: (1460, 81)
Test data: (1459, 80)
Final model trained successfully!

Submission file saved: /kaggle/working/submission.csv
Submission shape: (1459, 2)
Columns: ['Id', 'SalePrice']


,Id,SalePrice
0,1461,113369.179213
1,1462,148318.221520
2,1463,173424.022686
3,1464,194134.755769
4,1465,197282.721162



Submission checks passed!


In [2]:
import os

print(os.listdir("/kaggle/working"))

['__notebook__.ipynb', 'submission.csv']
